# Bodies, status codes and errors

## The body of a request: a Pydantic model

`POST` and `PUT` send data in their body, as JSON. Its shape is described by
a class inheriting from `BaseModel`, written like a dataclass.

In [1]:
from pydantic import BaseModel


class VenueIn(BaseModel):
    name: str
    capacity: int
    address: str = ""


body = VenueIn(name="Le Kiosque", capacity="120")
print(body)
print(body.capacity + 1)
print(body.model_dump())

name='Le Kiosque' capacity=120 address=''
121
{'name': 'Le Kiosque', 'capacity': 120, 'address': ''}


Unlike a dataclass, the annotations **are checked**: `"120"` became `120`.
A value that cannot be converted is refused:

In [2]:
VenueIn(name="Le Kiosque", capacity="many")

ValidationError: 1 validation error for VenueIn
capacity
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='many', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

## A route that creates something

In [3]:
from dataclasses import dataclass

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient


@dataclass
class Venue:
    name: str
    capacity: int

    def __post_init__(self):
        if self.capacity < 1:
            raise ValueError(f"A venue needs at least one seat, got {self.capacity}.")


app = FastAPI()
client = TestClient(app)
venues: dict[str, Venue] = {}


@app.post("/venues", status_code=201)
def create_venue(body: VenueIn) -> dict:
    if body.name in venues:
        raise HTTPException(status_code=409, detail=f"{body.name} already exists.")
    venue = Venue(body.name, body.capacity)
    venues[venue.name] = venue
    return {"name": venue.name, "capacity": venue.capacity}


@app.delete("/venues/{name}", status_code=204)
def delete_venue(name: str) -> None:
    if name not in venues:
        raise HTTPException(status_code=404, detail=f"Unknown venue: {name}.")
    del venues[name]


response = client.post("/venues", json={"name": "Le Kiosque", "capacity": 120})
print(response.status_code, response.json())

201 {'name': 'Le Kiosque', 'capacity': 120}


In [4]:
response = client.post("/venues", json={"name": "Le Kiosque", "capacity": 80})
print(response.status_code, response.json())
print(client.post("/venues", json={"name": "Le Kiosque"}).status_code)
print(client.delete("/venues/Le Kiosque").status_code)

409 {'detail': 'Le Kiosque already exists.'}
422
204


A parameter annotated with a `BaseModel` is read from the body.
`status_code=201` replaces the default `200` on success; `204` sends no body.

## The errors of the classes

What if the capacity is 0? `Venue` raises a `ValueError`, and nobody catches
it: for FastAPI it is a bug of the server.

In [5]:
client = TestClient(app, raise_server_exceptions=False)
response = client.post("/venues", json={"name": "Nowhere", "capacity": 0})
print(response.status_code, response.text)

500 Internal Server Error


The rule belongs to the class, not to the route: the route must not check the
capacity a second time. An **exception handler** turns an error of the classes
into an HTTP answer, for every route at once. It is declared with the routes,
before the application answers its first request:

In [6]:
from fastapi import Request
from fastapi.responses import JSONResponse

app = FastAPI()


@app.exception_handler(ValueError)
def handle_value_error(request: Request, error: ValueError) -> JSONResponse:
    return JSONResponse(status_code=422, content={"detail": str(error)})


@app.post("/venues", status_code=201)
def create_venue(body: VenueIn) -> dict:
    venue = Venue(body.name, body.capacity)
    return {"name": venue.name, "capacity": venue.capacity}


client = TestClient(app)
response = client.post("/venues", json={"name": "Nowhere", "capacity": 0})
print(response.status_code, response.json())

422 {'detail': 'A venue needs at least one seat, got 0.'}


In the project, every error inherits from `FestivalError`: one handler, and
a table from error class to status code (`NotFoundError` → 404,
`ScheduleConflictError` → 409...).

## An application built by a function

`venues` above is a global variable: every test would share it. The project
builds its application in a function, `create_app`, which receives what the
routes work on. Each call gives a new application with its own data.

In [7]:
def create_app(names: list[str]) -> FastAPI:
    app = FastAPI()

    @app.get("/names")
    def list_names() -> list[str]:
        return names

    @app.post("/names", status_code=201)
    def add_name(name: str) -> list[str]:
        names.append(name)
        return names

    return app


first = TestClient(create_app([]))
second = TestClient(create_app(["Nora"]))
first.post("/names", params={"name": "Max"})
print(first.get("/names").json())
print(second.get("/names").json())

['Max']
['Nora']


The routes are defined inside `create_app`: they see its parameter, like any
nested function.

| Layer | Knows | Example |
|---|---|---|
| classes | the rules | `Festival.add_performance` raises `ScheduleConflictError` |
| routes | HTTP | read the request, call the classes, answer |
| handler | both | `ScheduleConflictError` → `409` |